# Combine Daily Series

In [1]:
import numpy as np
import glob
import os
import pandas as pd
from collections import defaultdict

In [6]:
def working_days_series(year):
    dates = pd.date_range(start=f"{year}-01-01", end=f"{year}'-12-31", freq="D")
    series = pd.Series(np.where(dates.weekday < 5, 0, np.nan), index=dates)
    return series.dropna()

display(working_days_series(2024))

2024-01-01    0.0
2024-01-02    0.0
2024-01-03    0.0
2024-01-04    0.0
2024-01-05    0.0
             ... 
2024-12-25    0.0
2024-12-26    0.0
2024-12-27    0.0
2024-12-30    0.0
2024-12-31    0.0
Length: 262, dtype: float64

In [9]:
#path_template = "/media/dell/548d4dc2-4844-4adb-8b24-7b412b8f3455/0DT_Settelment_Backtest/HFT-Options-EIS-Global/tradelib/outputs/all_years/[YEAR]/trial_verification/delta_threshold/[DAY]__SPXW_DAY_OF_WEEK_DELTA_CONDOR_STRATEGY/consolidated_store"
path_template = "/media/dell/548d4dc2-4844-4adb-8b24-7b412b8f3455/0DT_Settelment_Backtest/HFT-Options-EIS-Global/tradelib/outputs/asia hedging/2024/all_day_hedging/[DAY]_10SPXW_DAY_OF_WEEK_DELTA_CONDOR_STRATEGY/consolidated_store/"

daily_pnl_dict = defaultdict(lambda:0)
weekly_pnl_dict = defaultdict(lambda:0)
monthly_pnl_dict = defaultdict(lambda:0)

for year in [2024]:
    daily_pnl = working_days_series(year)
    for day in ['MON','TUE','WED','THU','FRI']:
        path = path_template.replace('[YEAR]',str(year)).replace('[DAY]',day)
        file_names = np.array(glob.glob(pathname=os.path.join(path, "*.csv")))
        file_names.sort(kind="stable")  # Stable sort, optimized for almost sorted data

        # Concatenate CSV files into a single DataFrame
        backtest_df = pd.concat([pd.read_csv(file_path) for file_path in file_names], axis=0)
        backtest_df['timestamp'] = pd.to_datetime(backtest_df['timestamp'])
        backtest_df.set_index('timestamp',inplace = True)
        daily_series = backtest_df.loc[backtest_df["trade_done"],"portfolio_value"].resample('D').last().dropna().diff()
        
        daily_pnl = daily_pnl.add(daily_series,fill_value=0)
        
        print(year,day)
        print(daily_series.sum(),daily_series.isna().sum())
        
        if day == 'WED':
            display(daily_series.cumsum())
        
    daily_pnl_dict[year] = daily_pnl.dropna()
    weekly_pnl_dict[year] = daily_pnl.resample('W').sum().dropna()
    #monthly_pnl_dict[year] = daily_pnl.resample('M').sum().dropna()

2024 MON
105004.81600000178 1
2024 TUE
118215.71199999539 1
2024 WED
48807.68799999369 1


timestamp
2024-01-02           NaN
2024-01-03     -7017.032
2024-01-04     -7017.032
2024-01-05     -7017.032
2024-01-08     -7017.032
                 ...    
2024-12-12    185815.974
2024-12-13    185815.974
2024-12-16    185815.974
2024-12-17    175947.825
2024-12-18     48807.688
Name: portfolio_value, Length: 239, dtype: float64

2024 THU
45577.91199995891 1
2024 FRI
382752.8600000175 1


In [40]:
for key,value in weekly_pnl_dict.items():
    display(key)
    display(value.sum())
    display(value.min())

2025

2607776.3530000313

-290554.32800000685

In [41]:
for key,value in monthly_pnl_dict.items():
    display(key)
    display(value.sum())
    display(value.min())

In [42]:
daily_pnl_dict[2023].to_csv('plot.csv')

AttributeError: 'int' object has no attribute 'to_csv'

In [11]:
daily_pnl_dict[2024].to_csv('plot_2.csv')

In [72]:
series = (daily_pnl_dict[2025])

In [73]:
sum(series.loc["2025-05-19":'2025-06-12'])

433659.27399999835

In [80]:
real_pnl = {}
real_pnl["2025-05-19"]=1426
real_pnl["2025-05-20"]=2720
real_pnl["2025-05-21"]=-6880
real_pnl["2025-05-22"]=3534
real_pnl["2025-05-23"]=-5991
real_pnl["2025-05-27"]=2660
real_pnl["2025-05-28"]=2735
real_pnl["2025-05-29"]=1498
real_pnl["2025-05-30"]=696
real_pnl["2025-06-02"]=1273
real_pnl["2025-06-03"]=4197
real_pnl["2025-06-04"]=2949
real_pnl["2025-06-05"]=-4880
real_pnl["2025-06-06"]=842
real_pnl["2025-06-09"]=1709
real_pnl["2025-06-10"]=1595
real_pnl["2025-06-11"]=2623
real_pnl["2025-06-12"]=3814
real_pnl["2025-06-13"]=-5161


In [81]:
real_series = pd.Series(real_pnl)
real_series.index=pd.to_datetime(real_series.index)

In [8]:
import plotly.graph_objects as go
import pandas as pd

#start_date = "2025-05-19"
#end_date = "2025-06-13"

pnl_series = daily_pnl_dict[2024]
#pnl_series = pnl_series.loc["2025-05-19":"2025-06-13"]

cumulative_pnl = pnl_series.cumsum()
#cum_real_pnl=real_series.cumsum()

# Create a line plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cumulative_pnl.index,
    y=cumulative_pnl.values,
    mode='lines+markers',
    name='Cumulative Backtest PnL',
    line=dict(color='royalblue'),
    marker=dict(size=4)
))



fig.update_layout(
    title=f'Backtest PnL vs Real PnL',
    xaxis_title='Date',
    yaxis_title='PnL',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


In [94]:
pnl_series = daily_pnl_dict[2025]
#pnl_series = pnl_series.loc["2025-05-19":"2025-06-13"]

pnl_series_norm=pnl_series.loc["2025-05-19":"2025-06-13"]/10


diff = (real_series/pnl_series_norm-1)*100

fig = go.Figure(data=[go.Bar( x=diff.index,y=diff.values)])
fig.update_layout(
    title=f'Real PnL - Backtest Pnl difference percentage ({start_date} to {end_date})',
    xaxis_title='Date',
    yaxis_title='$',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [63]:
cum_real_pnl

2025-05-19     1426
2025-05-20     3770
2025-05-21    -3769
2025-05-22     -746
2025-05-23     5642
2025-05-27     8187
2025-05-28    10698
2025-05-29    11893
2025-05-30    12397
2025-06-02    13273
2025-06-03    17071
2025-06-04    19772
2025-06-05    14349
2025-06-06    14782
2025-06-09    16216
2025-06-10    17583
2025-06-11    19920
2025-06-12    23512
2025-06-13    17630
dtype: int64

In [64]:
real_pnl


{'2025-05-19': 1426,
 '2025-05-20': 2344,
 '2025-05-21': -7539,
 '2025-05-22': 3023,
 '2025-05-23': 6388,
 '2025-05-27': 2545,
 '2025-05-28': 2511,
 '2025-05-29': 1195,
 '2025-05-30': 504,
 '2025-06-02': 876,
 '2025-06-03': 3798,
 '2025-06-04': 2701,
 '2025-06-05': -5423,
 '2025-06-06': 433,
 '2025-06-09': 1434,
 '2025-06-10': 1367,
 '2025-06-11': 2337,
 '2025-06-12': 3592,
 '2025-06-13': -5882}